# KVLCC2 double-body surface flow

This notebook solves the unrestricted, infinite-depth surge potential on the public Gothenburg 2010 KVLCC2 geometry. MarineHydro uses $x>0$ toward the bow and the body-relative incoming stream is $-U\boldsymbol{e}_x$. The complete tangential velocity is therefore plotted, not just the perturbation-potential gradient. No free-surface plane or elevation field is rendered.

In [ ]:
using Revise
using Pkg

function marinehydro_root(start=pwd())
    directory = abspath(start)
    while true
        project = joinpath(directory, "Project.toml")
        if isfile(project) && occursin("name = \"MarineHydro\"", read(project, String))
            return directory
        end
        parent = dirname(directory)
        parent == directory && error("Run Jupyter from the MarineHydro.jl repository.")
        directory = parent
    end
end

project_root = marinehydro_root()
Pkg.activate(project_root)
using MarineHydro
using CairoMakie
using LinearAlgebra: norm
CairoMakie.activate!()

## Solve at the higher visualization resolution

`shape = (32, 17)` retains endpoint stations and produces 1,984 quadrilateral panels after mirroring. Reduce it only for exploratory development; the saved reference visualization uses the value below.

In [ ]:
shape = (32, 17)
U = 1.0
data_directory = joinpath(project_root, "validation", "gothenburg2010", "data", "KVLCC2")
paths = [
    joinpath(data_directory, "kvlcc_bow1.dat"),
    joinpath(data_directory, "kvlcc2_stn1.dat"),
]
all(isfile, paths) || error("Run validation/gothenburg2010/fetch_geometry.sh first.")

grid = read_gothenburg2010_panel_grid(paths; target_shape=shape)
mesh = grid.mesh
surge = solve_rigid_body_potential(mesh, :surge)
edge_velocity = body_relative_edge_velocity(
    mesh, U, 0.0, 0.0; surge_gradient=surge.potential_gradient,
)
speed = [norm(@view edge_velocity[panel, :]) for panel in 1:mesh.nfaces]

longitudinal_extent = extrema(mesh.centers[:, 1]) |> extrema -> extrema[2] - extrema[1]
bow_limit = maximum(mesh.centers[:, 1]) - 0.12 * longitudinal_extent
stagnation = surface_stagnation_panel(
    mesh, edge_velocity; panel_mask=mesh.centers[:, 1] .>= bow_limit,
)
streamlines = trace_surface_streamlines(
    mesh, edge_velocity, [first(strip) for strip in grid.strips];
    step_size=0.003 * longitudinal_extent,
    max_steps=650,
    neighbor_count=10,
    maximum_surface_distance=0.06 * longitudinal_extent,
)

(; panels=mesh.nfaces, strips=length(grid.strips),
   boundary_residual=surge.boundary_residual,
   stagnation_point=stagnation.point,
   stagnation_speed_ratio=stagnation.speed / U)

In [ ]:
function panel_wireframe(mesh)
    x = Float64[]; y = Float64[]; z = Float64[]
    for panel in 1:mesh.nfaces
        indices = mesh.faces[panel, :] .+ 1
        for index in (indices[1], indices[2], indices[3], indices[4], indices[1])
            push!(x, mesh.vertices[index, 1])
            push!(y, mesh.vertices[index, 2])
            push!(z, mesh.vertices[index, 3])
        end
        push!(x, NaN); push!(y, NaN); push!(z, NaN)
    end
    return x, y, z
end

wire_x, wire_y, wire_z = panel_wireframe(mesh)
x_bounds = extrema(mesh.vertices[:, 1])
y_bounds = extrema(mesh.vertices[:, 2])
z_bounds = extrema(mesh.vertices[:, 3])
function add_hull_flow!(axis; x_limits=x_bounds, show_stagnation=false)
    lines!(axis, wire_x, wire_y, wire_z; color=(:gray25, 0.28), linewidth=0.45)
    scatter!(axis, mesh.centers[:, 1], mesh.centers[:, 2], mesh.centers[:, 3];
        color=speed ./ U, colormap=:viridis, colorrange=(0, 1.35), markersize=3.0)
    for line in streamlines
        size(line.points, 1) < 2 && continue
        lines!(axis, line.points[:, 1], line.points[:, 2], line.points[:, 3];
            color=:orangered2, linewidth=1.8)
        scatter!(axis, [line.points[1, 1]], [line.points[1, 2]], [line.points[1, 3]];
            color=:gold, markersize=4)
    end
    if show_stagnation
        scatter!(axis, [stagnation.point[1]], [stagnation.point[2]], [stagnation.point[3]];
            color=:red, marker=:star5, markersize=18, strokecolor=:white, strokewidth=1)
    end
    xlims!(axis, x_limits...)
    ylims!(axis, y_bounds...)
    zlims!(axis, 1.12 * z_bounds[1], 0.012)
    return axis
end

## Orientation and close-up flow views

Gold dots are seeds, orange curves follow the complete body-relative surface velocity toward decreasing $x$, and the red star marks the lowest-speed panel in the forward 12% of the hull. The panel-center colors show $|\boldsymbol{u}_e|/U$.

In [ ]:
figure = Figure(size=(1800, 650), backgroundcolor=:white)
full_axis = Axis3(figure[1, 1]; title="Full KVLCC2: bow is +x, flow is toward -x",
    xlabel="x/Lpp", ylabel="y/Lpp", zlabel="z/Lpp", aspect=:data,
    azimuth=1.18pi, elevation=0.16pi)
bow_axis = Axis3(figure[1, 2]; title="Bow stagnation region",
    xlabel="x/Lpp", ylabel="y/Lpp", zlabel="z/Lpp", aspect=:data,
    azimuth=1.20pi, elevation=0.13pi)
stern_axis = Axis3(figure[1, 3]; title="Stern surface streamlines",
    xlabel="x/Lpp", ylabel="y/Lpp", zlabel="z/Lpp", aspect=:data,
    azimuth=1.20pi, elevation=0.13pi)

add_hull_flow!(full_axis; show_stagnation=true)
add_hull_flow!(bow_axis; x_limits=(x_bounds[2] - 0.18 * longitudinal_extent, x_bounds[2]),
    show_stagnation=true)
add_hull_flow!(stern_axis; x_limits=(x_bounds[1], x_bounds[1] + 0.20 * longitudinal_extent))

Label(figure[0, :], "KVLCC2 unrestricted infinite-depth double-body flow"; fontsize=24, font=:bold)
output_directory = joinpath(project_root, "validation", "gothenburg2010", "results")
mkpath(output_directory)
output_path = joinpath(output_directory, "kvlcc2_surface_flow_32x17.png")
save(output_path, figure; px_per_unit=1.5)
figure

## Interpretation

The bow lies at positive $x$. A physically consistent solution has the incoming surface flow moving from the positive-$x$ bow toward the negative-$x$ stern, while the lowest-speed forebody panel lies near the stem. The calculation is Wang's unrestricted double-body limit. Wave elevation would require a steady finite-Froude-number free-surface Green function or an equivalent nonlinear free-surface solve and is intentionally not visualized here.